# Batch Evaluation and Metrics

Simulate batch results offline, then optionally run live batch evaluation.

In [ ]:
import sys
from pathlib import Path

repo_src = Path("..").resolve() / "src"
if repo_src.exists():
    sys.path.insert(0, str(repo_src))

import pandas as pd
from openjury.output_format import AgentEvalResult, CriterionEvaluation, TrialResult
from openjury.scoring import JurorScore, ScoredMetrics

In [ ]:
# Simulate batch results (offline) and aggregate with BatchRunSummary
from openjury import EvaluationItem, ItemEvalResult, aggregate_batch_results
from openjury.execution import EvalItemStatus


def make_item(case_id: str, score: float, agreement: float, *, passed: bool = True) -> ItemEvalResult:
    metrics = ScoredMetrics(
        weighted_mean=score,
        mean=score,
        median=score,
        min_score=score - 0.3,
        max_score=score + 0.3,
        harmonic_mean=score - 0.1,
        weakest_link=score - 0.5,
        juror_agreement=agreement,
    )
    result = AgentEvalResult(
        jury_name="Notebook Jury",
        prompt=f"Prompt for {case_id}",
        score_scale=5,
        composite_score=score,
        normalized_composite_score=score / 5,
        scored_metrics=metrics,
        passed=passed,
        quality_passed=passed,
        assertion_threshold_met=True,
        quality_threshold=3.5,
        contested=agreement < 0.8,
        item_id=case_id,
    )
    return ItemEvalResult(
        item=EvaluationItem(prompt=result.prompt, item_id=case_id),
        index=0,
        result=result,
        status=EvalItemStatus.SCORED,
        evaluation_duration_ms=1200,
    )


item_results = [
    make_item("case-1", 4.2, 0.91),
    make_item("case-2", 3.1, 0.72, passed=False),
    make_item("case-3", 4.8, 0.95),
    make_item("case-4", 2.9, 0.65, passed=False),
]
summary = aggregate_batch_results(item_results, score_scale=5, score_min=1)
summary

In [ ]:
# Headline metrics for a SaaS run dashboard
print(f"Mean score: {summary.mean_composite_score:.2f} across {summary.scored_item_count} items")
print(f"Pass rate: {summary.pass_rate:.0%} ({summary.passed_count}/{summary.scored_item_count})")
print(f"Mean agreement: {summary.mean_juror_agreement:.2f}")
print(f"Contested cases: {summary.contested_count}")
print(f"P10 composite: {summary.score_distribution.p10:.2f}")
print("Histogram:", [(b.label, b.count) for b in summary.score_distribution.histogram])

In [ ]:
# Per-item table for debugging
rows = []
for item_result in item_results:
    result = item_result.result
    assert result is not None
    rows.append(
        {
            "case_id": result.item_id,
            "composite_score": result.composite_score,
            "passed": result.passed,
            "contested": result.contested,
            "juror_agreement": result.scored_metrics.juror_agreement,
        }
    )
pivot = pd.DataFrame(rows).set_index("case_id")
pivot.style.background_gradient(cmap="RdYlGn", subset=["composite_score", "juror_agreement"])

In [ ]:
# Optional: live batch with mock agent + OPENAI_API_KEY
import os
from openjury import EvaluationItem, ExecutionOptions, JuryConfig, OpenJury
from openjury.endpoint_fetcher import load_endpoints_file

if os.environ.get("OPENAI_API_KEY"):
    cfg = JuryConfig.from_json_file("../examples/basic_usage/config.json")
    endpoint = load_endpoints_file("../examples/basic_usage/endpoints.json")[0]
    jury = OpenJury(cfg)
    items = [
        EvaluationItem(prompt="How do I reset my password?", item_id="nb-1"),
        EvaluationItem(prompt="How do I cancel my subscription?", item_id="nb-2"),
    ]
    batch = jury.evaluate_items_with_summary(
        items,
        endpoint,
        options=ExecutionOptions(max_item_workers=2),
    )
    print("Pass rate:", batch.summary.pass_rate)
    print("Duration ms:", batch.duration_ms)
    live_rows = [
        {
            "case_id": r.item.item_id,
            "status": r.status.value,
            "composite_score": r.result.composite_score if r.result else None,
            "passed": r.result.passed if r.result else None,
            "contested": r.result.contested if r.result else None,
            "error_code": r.error_code,
        }
        for r in batch.items
    ]
    pd.DataFrame(live_rows)
else:
    print("Set OPENAI_API_KEY and start mock agent on :8080 for live batch eval.")